In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/rogii-wellbore-geology-prediction/sample_submission.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/AI_wellbore_geology_prediction_task_en.pptx
/kaggle/input/competitions/rogii-wellbore-geology-prediction/test/000d7d20__typewell.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/test/00e12e8b__typewell.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/test/00bbac68__typewell.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/test/000d7d20__horizontal_well.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/test/00bbac68__horizontal_well.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/test/00e12e8b__horizontal_well.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/train/75d20a82__typewell.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/train/b0f53bf4__horizontal_well.csv
/kaggle/input/competitions/rogii-wellbore-geology-prediction/train/e47

# ROGII — Wellbore Geology Prediction: Hybrid Viterbi + Linear-Tree Geosteering Pipeline

**Task**: predict `TVT` (true vertical thickness relative to the local structural datum) for the masked "Toe" section of each horizontal well, given its Gamma-Ray (GR) log, XYZ wellpath, and a reference vertical `typewell` GR/TVT profile.

**Architecture** (per the project's Advanced_Deep_Dive research docs):
1. **Feature engineering** — Q3D tortuosity/dogleg severity from the wellpath, azimuth, and a sliding-window distance-correlation match between the horizontal GR and the typewell GR.
2. **Stage 1 — Viterbi tracker**: a raw-GR (never z-scored) dynamic-programming sequence aligner, hard-anchored at the known Heel-end `TVT_input`, producing a physics-based baseline TVT path.
3. **Stage 2 — Linear-Tree residual model**: a regional structural dip plane plus a `LinearTreeRegressor` trained on `TVT - viterbi_tvt` residuals, using spatial (X, Y) leaf-linear extrapolation so predictions aren't capped at training-set depth bounds.
4. **Defensive validation**: sequence-shuffle leakage test, no-op flat-line benchmark, and leave-spatial-out block CV — gated behind `RUN_DIAGNOSTICS` so the scored submission run stays fast.
5. **Submission**: dynamically globbed test wells, predictions only where `TVT_input` is NaN, `id = {WELLNAME}_{row_index}`.

Known real schema (confirmed against the actual Kaggle kernel's file listing and the project's dataset-schema doc):
- `{well}__horizontal_well.csv`: `MD, X, Y, Z, GR, TVT` (train) / `TVT_input` (test, NaN over the masked Toe), plus per-formation depth columns.
- `{well}__typewell.csv`: `TVT, GR, Geology`.
- No `inclination`/`azimuth` columns exist in the raw files — these are **derived** below from consecutive `MD, X, Y, Z` station deltas.
- Train has 773 wells (+ PNGs, unused — PNGs don't exist in test). Test is dynamically globbed since Kaggle swaps in a hidden set at scoring time.

In [3]:
# Setup: install linear-tree (internet is enabled on this kernel per user confirmation) and import core libraries.
import sys
import subprocess

try:
    from lineartree import LinearTreeRegressor
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "linear-tree"], check=True)
    from lineartree import LinearTreeRegressor

import glob
import warnings
from typing import Tuple, Optional, List, Dict

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.base import BaseEstimator

# Compatibility shim: scikit-learn >=1.6 removed the private BaseEstimator._validate_data
# method (moved to the module-level sklearn.utils.validation.validate_data) and renamed the
# `force_all_finite` kwarg to `ensure_all_finite`, but the `linear-tree` package (last released
# for older sklearn) still calls self._validate_data(..., force_all_finite=True). Without this,
# LinearTreeRegressor.fit raises AttributeError/TypeError on newer sklearn. Confirmed live
# against sklearn 1.9.0 locally (round-tripped through both failure modes); harmless no-op if
# the original method/kwarg still exist.
if not hasattr(BaseEstimator, "_validate_data"):
    from sklearn.utils.validation import validate_data as _sklearn_validate_data

    def _validate_data_shim(self, X, y=None, **kwargs):
        kwargs.pop("reset", None)
        if "force_all_finite" in kwargs:
            kwargs["ensure_all_finite"] = kwargs.pop("force_all_finite")
        if y is None:
            return _sklearn_validate_data(self, X, **kwargs)
        return _sklearn_validate_data(self, X, y, **kwargs)

    BaseEstimator._validate_data = _validate_data_shim

warnings.filterwarnings("ignore")
np.set_printoptions(suppress=True)

In [4]:
# Configuration

DATA_ROOT = "/kaggle/input/competitions/rogii-wellbore-geology-prediction"
TRAIN_DIR = f"{DATA_ROOT}/train"
TEST_DIR = f"{DATA_ROOT}/test"

RUN_DIAGNOSTICS = True          # Set False to skip Gate 1-3 validation and go straight to submission.
RANDOM_STATE = 42
N_JOBS = -1                     # Parallel workers for per-well feature/Viterbi construction. -1 = all cores.

# Viterbi state-space: a local band around the anchor rather than the full typewell range —
# TVT cannot plausibly jump further than the well's own vertical excursion within a single lateral,
# so bounding the state window keeps the O(T * M^2) dynamic program tractable across hundreds of wells.
VITERBI_STATE_STEP_FT = 0.5
VITERBI_STATE_HALFWIDTH_FT = 150.0

# Fraction of each training well's rows treated as the known "Heel" context when simulating the
# train/test split locally (test wells reveal roughly the first 20-30% of TVT_input before masking).
HEEL_FRACTION_RANGE = (0.15, 0.35)

FEATURE_COLS = [
    "X", "Y", "MD_from_ps", "GR", "tortuosity_cum",
    "azimuth_sin", "azimuth_cos", "dcor_gr_match", "viterbi_tvt",
]

In [5]:
def list_well_ids(split: str) -> List[str]:
    """Dynamically discover well IDs present in a split ('train' or 'test'), robust to the
    hidden test set being swapped in at scoring time."""
    directory = TRAIN_DIR if split == "train" else TEST_DIR
    paths = sorted(glob.glob(f"{directory}/*__horizontal_well.csv"))
    return [os.path.basename(p).replace("__horizontal_well.csv", "") for p in paths]


def load_horizontal(well_id: str, split: str) -> pd.DataFrame:
    directory = TRAIN_DIR if split == "train" else TEST_DIR
    df = pd.read_csv(f"{directory}/{well_id}__horizontal_well.csv")
    df["well_id"] = well_id
    # Preserve the original row position (0..n-1 as read from disk) — the submission ID
    # format `{WELLNAME}_{row_index}` refers to THIS index, which must survive any later
    # sort/dedup so predictions map back to the correct id.
    df["orig_index"] = df.index
    return df


def load_typewell(well_id: str, split: str) -> pd.DataFrame:
    directory = TRAIN_DIR if split == "train" else TEST_DIR
    df = pd.read_csv(f"{directory}/{well_id}__typewell.csv")
    df = df.dropna(subset=["TVT", "GR"]).sort_values("TVT").reset_index(drop=True)
    return df


train_well_ids = list_well_ids("train")
test_well_ids = list_well_ids("test")
print(f"Train wells found: {len(train_well_ids)}")
print(f"Test wells found (dynamically globbed, may be a hidden set at scoring time): {len(test_well_ids)}")

Train wells found: 773
Test wells found (dynamically globbed, may be a hidden set at scoring time): 3


## Preprocessing: resolution alignment + spike-noise filtering

Horizontal GR logs are sampled at survey-station resolution and can carry sensor spikes. We
apply a rolling-median filter (odd window, edge-safe) to the raw horizontal GR before it
feeds either the tortuosity/dcor feature extraction or the Viterbi emission cost — the
Viterbi tracker itself must still see **raw API-unit** GR (never z-scored), per the
Z-Score Scaling Pitfall documented in the research notes.

In [6]:
def rolling_median_denoise(gr: np.ndarray, window: int = 5) -> np.ndarray:
    """Edge-safe rolling-median spike filter. `window` must be odd."""
    if window % 2 == 0:
        window += 1
    s = pd.Series(gr)
    denoised = s.rolling(window=window, center=True, min_periods=1).median()
    return denoised.to_numpy()


def preprocess_horizontal(df: pd.DataFrame) -> pd.DataFrame:
    """Sort by MD, drop duplicate stations, interpolate small GR gaps, denoise spikes."""
    df = df.sort_values("MD").drop_duplicates(subset="MD").reset_index(drop=True)
    df["GR"] = df["GR"].interpolate(limit_direction="both")
    df["GR_raw"] = df["GR"].to_numpy()
    df["GR"] = rolling_median_denoise(df["GR"].to_numpy(), window=5)
    for col in ("X", "Y", "Z"):
        df[col] = df[col].interpolate(limit_direction="both")
    return df

## Feature engineering 1/3 — Derived inclination/azimuth, Q3D tortuosity, azimuth normalization

The raw files only give `MD, X, Y, Z`, not the `inclination`/`azimuth` survey angles the Q3D
tortuosity formula is defined over. We invert the Minimum Curvature relationship locally:
for each station-to-station step, `inclination = arccos(-ΔZ / ΔMD)` (`Z` is elevation,
negative-down, so `-ΔZ` is the true-vertical component of the step) and
`azimuth = atan2(ΔX, ΔY) mod 360` (compass convention, 0°=North/+Y, 90°=East/+X — the same
convention used for well-level azimuth in the validation-fold code). Tortuosity is then the
dogleg-severity / Ratio-Factor formula applied to these derived angles, exactly as specified.

In [7]:
def derive_inclination_azimuth(df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    """Invert the Minimum Curvature relationship locally to recover per-station
    inclination/azimuth from consecutive MD, X, Y, Z stations (not present as raw columns)."""
    md, x, y, z = df["MD"].to_numpy(), df["X"].to_numpy(), df["Y"].to_numpy(), df["Z"].to_numpy()
    n = len(md)
    inc_deg = np.zeros(n)
    az_deg = np.zeros(n)

    d_md = np.diff(md)
    d_md_safe = np.where(d_md <= 1e-6, np.nan, d_md)
    dx, dy, dz = np.diff(x), np.diff(y), np.diff(z)

    cos_inc = np.clip(-dz / d_md_safe, -1.0, 1.0)
    inc = np.degrees(np.arccos(np.nan_to_num(cos_inc, nan=0.0)))
    az = np.degrees(np.arctan2(dx, dy)) % 360.0

    inc_deg[1:] = inc
    inc_deg[0] = inc[0] if n > 1 else 0.0
    az_deg[1:] = az
    az_deg[0] = az[0] if n > 1 else 0.0
    return inc_deg, az_deg


def calculate_q3d_tortuosity(md: np.ndarray, inc_deg: np.ndarray, az_deg: np.ndarray) -> np.ndarray:
    """Q3D tortuosity via spherical dogleg-severity + Ratio-Factor correction, cumulatively summed.
    Adapted from the project's `calculate_q3d_tortuosity` reference implementation."""
    inc = np.radians(inc_deg)
    az = np.radians(az_deg)
    n = len(md)
    local_tort = np.zeros(n)

    for i in range(1, n):
        d_md = md[i] - md[i - 1]
        if d_md <= 1e-6:
            continue
        cos_gamma = (np.cos(inc[i - 1]) * np.cos(inc[i]) +
                     np.sin(inc[i - 1]) * np.sin(inc[i]) * np.cos(az[i] - az[i - 1]))
        cos_gamma = np.clip(cos_gamma, -1.0, 1.0)
        gamma = np.arccos(cos_gamma)
        rf = (2.0 / gamma) * np.tan(gamma / 2.0) if gamma > 1e-6 else 1.0
        local_tort[i] = (gamma * rf) / d_md

    return np.cumsum(local_tort)


def well_azimuth_sin_cos(df: pd.DataFrame) -> Tuple[float, float]:
    """Well-level drilling azimuth from start/end coordinates (compass convention), returned
    as (sin, cos) so downstream models don't have to deal with the 0/360 discontinuity."""
    dx = df["X"].iloc[-1] - df["X"].iloc[0]
    dy = df["Y"].iloc[-1] - df["Y"].iloc[0]
    azimuth_deg = np.degrees(np.arctan2(dx, dy)) % 360.0
    az_rad = np.radians(azimuth_deg)
    return np.sin(az_rad), np.cos(az_rad)

## Feature engineering 2/3 — Sliding distance correlation (`dcor_sliding`)

Pearson correlation is ≈0 for cyclically warped-but-matching GR shapes (a horizontal well
crossing the same shale/sand cycle at a different phase/frequency than the typewell still
"matches" geologically). Distance correlation captures nonlinear/shape association and is
scale-invariant. We slide a window over the horizontal GR log and, for each window, compute
the distance correlation against the *whole* typewell GR profile — the max dCor across a
coarse TVT-anchor sweep becomes a per-row "how well does local GR shape match the reference
profile" feature.

In [8]:
def _distance_correlation(a: np.ndarray, b: np.ndarray) -> float:
    """Distance correlation (Szekely & Rizzo) between two equal-length 1D arrays."""
    n = len(a)
    if n < 4:
        return 0.0
    A = np.abs(a[:, None] - a[None, :])
    B = np.abs(b[:, None] - b[None, :])
    A = A - A.mean(axis=0, keepdims=True) - A.mean(axis=1, keepdims=True) + A.mean()
    B = B - B.mean(axis=0, keepdims=True) - B.mean(axis=1, keepdims=True) + B.mean()
    dcov2 = (A * B).mean()
    dvar_a = (A * A).mean()
    dvar_b = (B * B).mean()
    denom = np.sqrt(max(dvar_a, 0.0) * max(dvar_b, 0.0))
    if denom < 1e-12:
        return 0.0
    return float(np.sqrt(max(dcov2, 0.0) / denom))


def dcor_sliding_feature(
    lateral_gr: np.ndarray,
    typewell_gr: np.ndarray,
    window: int = 11,
    max_lateral_samples: int = 60,
    max_typewell_windows: int = 40,
) -> np.ndarray:
    """For a capped, evenly-spaced sample of lateral rows, find the best-matching same-length
    window in the typewell GR profile (also capped/evenly-spaced) by distance correlation, and
    return the per-row max dCor — a scale-invariant "does the local GR shape look like a real
    geological signature" feature — linearly interpolated back to full row resolution.

    Sample counts are capped (not just strided) regardless of well/typewell length: with 773
    training wells to process, an uncapped O(n_lateral * n_typewell) sweep at per-pair O(window^2)
    cost would take hours: max_lateral_samples * max_typewell_windows bounds every well to the
    same small, fixed number of dCor evaluations."""
    n = len(lateral_gr)
    half = window // 2
    valid_range = np.arange(half, max(n - half, half + 1))
    if len(valid_range) == 0:
        return np.zeros(n)
    sample_idx = np.unique(np.linspace(valid_range[0], valid_range[-1], num=min(max_lateral_samples, len(valid_range)), dtype=int))

    tw_valid_starts = np.arange(0, max(len(typewell_gr) - window, 1))
    if len(tw_valid_starts) == 0:
        return np.zeros(n)
    tw_starts = np.unique(np.linspace(tw_valid_starts[0], tw_valid_starts[-1], num=min(max_typewell_windows, len(tw_valid_starts)), dtype=int))

    scores = np.zeros(len(sample_idx))
    for k, i in enumerate(sample_idx):
        lat_win = lateral_gr[i - half: i - half + window]
        if len(lat_win) != window:
            continue
        best = 0.0
        for j in tw_starts:
            tw_win = typewell_gr[j: j + window]
            if len(tw_win) != window:
                continue
            d = _distance_correlation(lat_win, tw_win)
            if d > best:
                best = d
        scores[k] = best

    return np.interp(np.arange(n), sample_idx, scores)

## Stage 1 — Viterbi tracker (raw-GR, heel-anchored, vectorized, windowed state space)

This adapts the project's `AdvancedViterbiTracker` reference design with three deliberate
changes, each noted because they diverge from the literal source docs:

1. **Fixes a latent bug** in the source `AdvancedViterbiTracker` — its code imports only
   `Union` from `typing` but its `_normalize_signals` signature references `Tuple`, which
   would raise `NameError` at call time. `Tuple` is imported in the setup cell above.
2. **Vectorizes the state loop.** The reference implementation loops over states in Python
   for every timestep (`for s in range(self.n_states)`), which is O(T·M²) at the Python
   level. We replace that with a fully broadcasted `(M, M)` transition-cost matrix per
   timestep, leaving only a Python loop over T — the same recurrence, computed faster.
3. **Bounds the state space to a local window** around the anchor (`VITERBI_STATE_HALFWIDTH_FT`)
   instead of the full typewell TVT range, and **anchors at the Heel/Toe boundary** (using
   the last known Heel `TVT`/`TVT_input` value) rather than at row 0 of the file — the Heel
   rows themselves are already known and don't need tracking; only the masked Toe does.

In [9]:
class AdvancedViterbiTracker:
    """Raw-GR Viterbi sequence aligner, heel-anchored, with a bounded local state window
    and a vectorized (T-loop only, no per-state Python loop) forward pass."""

    def __init__(self, typewell_tvt: np.ndarray, typewell_gr: np.ndarray,
                 anchor_tvt: float, state_step: float = VITERBI_STATE_STEP_FT,
                 state_halfwidth: float = VITERBI_STATE_HALFWIDTH_FT):
        lo = anchor_tvt - state_halfwidth
        hi = anchor_tvt + state_halfwidth
        self.states = np.arange(lo, hi + state_step, state_step)
        self.state_gr = np.interp(self.states, typewell_tvt, typewell_gr)
        self.n_states = len(self.states)

        gr_mean = float(np.mean(typewell_gr))
        # lambda calibrated to raw-GR API units per the Z-Score Scaling Pitfall analysis:
        # lambda ~= mean(GR_raw)^2 / 100, clipped to the documented 20-80 band.
        self.lambda_param = float(np.clip((gr_mean ** 2) / 100.0, 20.0, 80.0))

    def track(self, lateral_gr: np.ndarray, delta_z: np.ndarray, anchor_tvt: float) -> np.ndarray:
        """lateral_gr / delta_z cover ONLY the segment to be tracked (the masked Toe);
        anchor_tvt is the known TVT at the row immediately before this segment starts."""
        T = len(lateral_gr)
        M = self.n_states
        dp = np.full((M, T), 1e6)
        path_ptr = np.zeros((M, T), dtype=int)

        anchor_idx = int(np.argmin(np.abs(self.states - anchor_tvt)))
        dp[anchor_idx, 0] = (self.state_gr[anchor_idx] - lateral_gr[0]) ** 2

        state_diff = self.states[:, None] - self.states[None, :]  # (M, M): s_i - s_j

        for t in range(1, T):
            dz = delta_z[t]
            transition_cost = self.lambda_param * (state_diff - dz) ** 2   # (M, M)
            total_cost = dp[None, :, t - 1] + transition_cost              # (M, M) over (i, j)
            best_prev = np.argmin(total_cost, axis=1)                      # (M,)
            path_ptr[:, t] = best_prev
            gr_match_cost = (self.state_gr - lateral_gr[t]) ** 2
            dp[:, t] = total_cost[np.arange(M), best_prev] + gr_match_cost

        opt_path = np.zeros(T, dtype=int)
        opt_path[-1] = int(np.argmin(dp[:, -1]))
        for t in range(T - 2, -1, -1):
            opt_path[t] = path_ptr[opt_path[t + 1], t + 1]

        return self.states[opt_path]

## Regional structural plane

A Ridge-regularized plane `TVT = a*X + b*Y + c` fit across training wells' spatial
coordinates, giving the regional dip trend used both as a sanity check on the Viterbi
baseline and, implicitly, as the kind of spatial signal the Linear-Tree residual model
exploits directly through its own `X, Y` leaf-linear features.

In [10]:
class RegionalPlaneFitter:
    """Structural dip plane: TVT = a*X + b*Y + c."""

    def __init__(self, alpha: float = 10.0):
        self.model = Ridge(alpha=alpha)

    def fit(self, x: np.ndarray, y: np.ndarray, tvt: np.ndarray) -> "RegionalPlaneFitter":
        features = np.column_stack((x, y))
        self.model.fit(features, tvt)
        return self

    def predict(self, x: np.ndarray, y: np.ndarray) -> np.ndarray:
        features = np.column_stack((x, y))
        return self.model.predict(features)

    def get_dip_vector(self) -> Tuple[float, float]:
        return float(self.model.coef_[0]), float(self.model.coef_[1])

## Per-well feature + Viterbi-baseline construction

`build_well_frame` is the single function used for **both** training and test wells, so the
exact same feature pipeline and Viterbi procedure sees both splits (the reference doc left
this step as a literal `...` placeholder for training wells — this fills that gap):

- **Test wells**: the Heel/Toe split is real — `TVT_input` is known pre-PS and `NaN` post-PS.
- **Train wells**: `TVT` is fully known, so we *simulate* a PS cut at a per-well deterministic
  fraction of rows (seeded from a hash of `well_id`, within `HEEL_FRACTION_RANGE`) to match the
  test-time Heel/Toe proportions — the Viterbi tracker only ever sees GR from the simulated
  Toe onward, exactly mirroring what it will face at inference, so the residual target
  (`TVT - viterbi_tvt`) is trained on a distribution that matches test time.

In [11]:
def _deterministic_heel_fraction(well_id: str) -> float:
    lo, hi = HEEL_FRACTION_RANGE
    h = int(hash(well_id) % 10_000) / 10_000.0
    return lo + h * (hi - lo)


def build_well_frame(well_id: str, split: str) -> Optional[pd.DataFrame]:
    """Load one well, engineer features, split Heel/Toe, run the Viterbi tracker on the Toe,
    and (for train) attach the tvt_residual training target. Returns None if the well is
    unusable (e.g. degenerate/empty typewell)."""
    horiz = preprocess_horizontal(load_horizontal(well_id, split))
    typewell = load_typewell(well_id, split)
    if len(horiz) < 5 or len(typewell) < 5:
        return None

    inc_deg, az_deg = derive_inclination_azimuth(horiz)
    horiz["tortuosity_cum"] = calculate_q3d_tortuosity(horiz["MD"].to_numpy(), inc_deg, az_deg)
    az_sin, az_cos = well_azimuth_sin_cos(horiz)
    horiz["azimuth_sin"] = az_sin
    horiz["azimuth_cos"] = az_cos
    horiz["dcor_gr_match"] = dcor_sliding_feature(horiz["GR"].to_numpy(), typewell["GR"].to_numpy())

    z = horiz["Z"].to_numpy()
    delta_z_full = np.diff(z, prepend=z[0])

    if split == "test":
        known_mask = horiz["TVT_input"].notna().to_numpy()
        horiz["TVT"] = horiz["TVT_input"]
    else:
        heel_frac = _deterministic_heel_fraction(well_id)
        cut = max(2, int(len(horiz) * heel_frac))
        known_mask = np.zeros(len(horiz), dtype=bool)
        known_mask[:cut] = True

    if not known_mask.any() or known_mask.all():
        return None

    toe_start = int(np.argmax(~known_mask))  # first False after the known prefix
    heel_end_tvt = float(horiz["TVT"].to_numpy()[toe_start - 1])

    toe_gr = horiz["GR"].to_numpy()[toe_start:]
    toe_delta_z = delta_z_full[toe_start:]

    tracker = AdvancedViterbiTracker(
        typewell_tvt=typewell["TVT"].to_numpy(),
        typewell_gr=typewell["GR"].to_numpy(),
        anchor_tvt=heel_end_tvt,
    )
    toe_viterbi = tracker.track(toe_gr, toe_delta_z, heel_end_tvt)

    viterbi_tvt = horiz["TVT"].to_numpy().astype(float).copy()
    viterbi_tvt[toe_start:] = toe_viterbi
    horiz["viterbi_tvt"] = viterbi_tvt
    horiz["is_toe"] = ~known_mask
    horiz["MD_from_ps"] = horiz["MD"] - horiz["MD"].to_numpy()[toe_start]

    if split == "train":
        horiz["tvt_residual"] = horiz["TVT"] - horiz["viterbi_tvt"]

    return horiz

<cell_type>markdown</cell_type>## Build the training feature set

Runs `build_well_frame` over every training well **in parallel** (via `joblib`, one process per
well, `N_JOBS` workers — each well's Viterbi tracking + feature extraction is fully
independent, so this is embarrassingly parallel) and keeps only the simulated-Toe rows —
those are the rows the residual model will actually be scored on at inference time (Heel
rows are never predicted, so training the residual model on them would mismatch the
deployment distribution).

In [12]:
def _build_well_frame_safe(well_id: str, split: str):
    """Wrapper that captures exceptions as return values instead of raising — an exception
    raised inside a joblib worker would otherwise abort the entire parallel batch."""
    try:
        return build_well_frame(well_id, split)
    except Exception as exc:
        return RuntimeError(f"{well_id}: {exc}")


def build_split_dataset(well_ids: List[str], split: str, n_jobs: int = N_JOBS) -> pd.DataFrame:
    results = Parallel(n_jobs=n_jobs, backend="loky", verbose=5)(
        delayed(_build_well_frame_safe)(well_id, split) for well_id in well_ids
    )

    frames, errors = [], []
    for result in results:
        if isinstance(result, Exception):
            errors.append(str(result))
        elif result is not None:
            frames.append(result)

    if errors:
        print(f"  [skipped {len(errors)} wells]:")
        for msg in errors[:10]:
            print(f"    {msg}")
        if len(errors) > 10:
            print(f"    ... and {len(errors) - 10} more")

    print(f"  processed {len(well_ids)} {split} wells "
          f"({len(frames)} usable, {len(errors)} skipped)")
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


train_full_df = build_split_dataset(train_well_ids, "train")
train_toe_df = train_full_df[train_full_df["is_toe"]].reset_index(drop=True)
print(f"Train wells processed: {train_full_df['well_id'].nunique()}")b 
print(f"Train Toe rows (training examples for the residual model): {len(train_toe_df)}")

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=-1)]: Done  10 tasks      | elapsed:   48.9s
[Parallel(n_jobs=-1)]: Done  64 tasks      | elapsed:  4.9min
[Parallel(n_jobs=-1)]: Done 154 tasks      | elapsed: 11.7min
[Parallel(n_jobs=-1)]: Done 280 tasks      | elapsed: 21.0min
[Parallel(n_jobs=-1)]: Done 442 tasks      | elapsed: 33.0min
[Parallel(n_jobs=-1)]: Done 640 tasks      | elapsed: 47.8min
[Parallel(n_jobs=-1)]: Done 773 out of 773 | elapsed: 57.6min finished


  processed 773 train wells (773 usable, 0 skipped)
Train wells processed: 773
Train Toe rows (training examples for the residual model): 3823616


In [13]:
# Fit the regional structural plane on ALL known TVT rows across training wells (Heel + Toe),
# giving the widest possible spatial coverage of the dip trend.
plane_fitter = RegionalPlaneFitter(alpha=10.0)
plane_fitter.fit(
    train_full_df["X"].to_numpy(),
    train_full_df["Y"].to_numpy(),
    train_full_df["TVT"].to_numpy(),
)
dip_x, dip_y = plane_fitter.get_dip_vector()
print(f"Regional dip vector: dTVT/dX={dip_x:.5f} ft/ft, dTVT/dY={dip_y:.5f} ft/ft")

Regional dip vector: dTVT/dX=0.02366 ft/ft, dTVT/dY=-0.02518 ft/ft


## Stage 2 — LinearTreeRegressor residual correction

Trained on `tvt_residual = TVT - viterbi_tvt` over the simulated-Toe training rows, using
`FEATURE_COLS` (spatial `X, Y` for out-of-training-bounds extrapolation, `MD_from_ps` for
distance-from-anchor decay, tortuosity/azimuth/dcor for structural-behavior context, raw GR,
and the Viterbi baseline itself so the tree can learn where that baseline tends to be
systematically off). `LinearRegression` leaves let the tree extrapolate the regional dip
trend linearly instead of flat-lining outside the training depth range.

In [ ]:
class LinearTreeExtrapolator:
    """Piecewise-linear residual corrector: LinearTreeRegressor with Ridge-regularized leaves,
    over standardized features.

    Root cause found via live dry run (nbconvert against synthetic data): FEATURE_COLS mixes
    wildly different scales (X, Y ~ thousands of feet; tortuosity_cum ~ 1e-3; azimuth_sin/cos
    in [-1, 1]). `LinearTreeRegressor` requires `base_estimator` to be a literal
    sklearn.linear_model instance (a Pipeline with StandardScaler is explicitly rejected:
    "Only linear models are accepted as base_estimator"), so Ridge alone — even with
    regularization — assigns huge coefficients to the tiny-scale features to make them
    influential, and those coefficients then explode when a leaf extrapolates to an
    out-of-distribution row (confirmed: Gate 3 leave-spatial-out RMSE was ~1e14 ft with
    Ridge(alpha=1.0) on raw features). Standardizing the feature matrix ourselves before it
    ever reaches LinearTreeRegressor keeps coefficients on a comparable scale and bounds
    extrapolation to sane magnitudes (verified: far out-of-range test rows now predict in the
    tens-of-feet range instead of 1e14)."""

    def __init__(self, max_depth: int = 5, min_samples_leaf: int = 50, ridge_alpha: float = 1.0):
        self.scaler = StandardScaler()
        self.model = LinearTreeRegressor(
            base_estimator=Ridge(alpha=ridge_alpha),
            max_depth=max_depth,
            min_samples_leaf=min_samples_leaf,
        )

    def fit(self, X: pd.DataFrame, y: np.ndarray) -> "LinearTreeExtrapolator":
        Xs = self.scaler.fit_transform(X.to_numpy())
        self.model.fit(Xs, y)
        return self

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        Xs = self.scaler.transform(X.to_numpy())
        return self.model.predict(Xs)


def clean_feature_matrix(df: pd.DataFrame, feature_cols: List[str]) -> pd.DataFrame:
    X = df[feature_cols].copy()
    return X.fillna(X.median(numeric_only=True))


train_features = clean_feature_matrix(train_toe_df, FEATURE_COLS)
train_target = train_toe_df["tvt_residual"].to_numpy()

residual_model = LinearTreeExtrapolator(max_depth=5, min_samples_leaf=50)
residual_model.fit(train_features, train_target)
print("LinearTreeRegressor re  sidual model trained on", len(train_features), "rows.")

LinearTreeRegressor residual model trained on 3823616 rows.


## Defensive validation (Gates 1–3)

Gated behind `RUN_DIAGNOSTICS` so the scored submission run isn't slowed down by them. All
three gates train/evaluate strictly on **training wells** (never touching test), using a
Leave-Spatial-Out split as the base fold structure so Gate 1 and Gate 2 are themselves
evaluated out-of-fold rather than on training-fitted rows.

- **Gate 3 (Leave-Spatial-Out)** clusters wells by their mean `(X, Y)` with K-Means into
  `K` blocks, and for each held-out block retrains the plane + residual model on the
  remaining blocks only, predicting on the held-out block — simulating an unseen fault block.
- **Gate 2 (No-op flat-line)** compares the trained model's out-of-fold RMSE against simply
  holding the Heel-end TVT constant across the whole Toe.
- **Gate 1 (Sequence shuffle)** reruns the Viterbi tracker with the Toe's GR/`ΔZ` sequence
  randomly permuted (destroying MD continuity while keeping each row's own X, Y, GR, and
  true TVT together) and checks that predictions degrade sharply rather than staying flat
  (a flat degradation would mean the residual model is leaning on `X, Y` coordinate leakage
  rather than genuine GR-sequence alignment).

In [ ]:
def rmse(pred: np.ndarray, true: np.ndarray) -> float:
    pred, true = np.asarray(pred, dtype=float), np.asarray(true, dtype=float)
    mask = np.isfinite(pred) & np.isfinite(true)
    return float(np.sqrt(np.mean((pred[mask] - true[mask]) ** 2))) if mask.any() else np.nan


if RUN_DIAGNOSTICS:
    print("=" * 70)
    print("GATE 3 — Leave-Spatial-Out Block CV")
    print("=" * 70)

    well_xy = (
        train_full_df.groupby("well_id")[["X", "Y"]].mean().reset_index()
    )
    N_SPATIAL_CLUSTERS = 8
    kmeans = KMeans(n_clusters=N_SPATIAL_CLUSTERS, random_state=RANDOM_STATE, n_init=10)
    well_xy["cluster"] = kmeans.fit_predict(well_xy[["X", "Y"]].to_numpy())
    well_to_cluster = dict(zip(well_xy["well_id"], well_xy["cluster"]))
    train_toe_df["cluster"] = train_toe_df["well_id"].map(well_to_cluster)

    oof_pred = np.full(len(train_toe_df), np.nan)

    heel_end_by_well = train_full_df.loc[~train_full_df["is_toe"]].groupby("well_id").apply(
        lambda g: g.sort_values("MD")["TVT"].iloc[-1]
    ).to_dict()
    oof_noop = train_toe_df["well_id"].map(heel_end_by_well).to_numpy()

    fold_models: Dict[int, LinearTreeExtrapolator] = {}
    for cluster_id in sorted(well_xy["cluster"].unique()):
        held_out_wells = set(well_xy.loc[well_xy["cluster"] == cluster_id, "well_id"])
        fold_train_mask = ~train_toe_df["well_id"].isin(held_out_wells)
        fold_val_mask = train_toe_df["well_id"].isin(held_out_wells)
        if fold_train_mask.sum() < 50 or fold_val_mask.sum() == 0:
            continue

        X_fold_train = clean_feature_matrix(train_toe_df.loc[fold_train_mask], FEATURE_COLS)
        y_fold_train = train_toe_df.loc[fold_train_mask, "tvt_residual"].to_numpy()
        fold_model = LinearTreeExtrapolator(max_depth=5, min_samples_leaf=50)
        fold_model.fit(X_fold_train, y_fold_train)
        fold_models[cluster_id] = fold_model

        X_fold_val = clean_feature_matrix(train_toe_df.loc[fold_val_mask], FEATURE_COLS)
        pred_residual = fold_model.predict(X_fold_val)
        pred_tvt = train_toe_df.loc[fold_val_mask   , "viterbi_tvt"].to_numpy() + pred_residual
        oof_pred[np.where(fold_val_mask.to_numpy())[0]] = pred_tvt

    true_tvt = train_toe_df["TVT"].to_numpy()
    rmse_model_oof = rmse(oof_pred, true_tvt)
    rmse_noop_oof = rmse(oof_noop, true_tvt)
    print(f"Out-of-fold model RMSE (Leave-Spatial-Out): {rmse_model_oof:.3f} ft")
    print(f"Out-of-fold no-op flat-line RMSE:            {rmse_noop_oof:.3f} ft")
    print(f"Gate 3 verdict: {'PASS' if rmse_model_oof < rmse_noop_oof else 'FAIL'} "
          f"(model must generalize across held-out spatial clusters)")

GATE 3 — Leave-Spatial-Out Block CV


In [ ]:
if RUN_DIAGNOSTICS:
    print("=" * 70)  
    print("GATE 2 — No-op Flat-Line Anchor-Hold Benchmark")
    print("=" * 70)
    delta = rmse_noop_oof - rmse_model_oof
    print(f"RMSE_model={rmse_model_oof:.3f} ft vs RMSE_no-op={rmse_noop_oof:.3f} ft "
          f"(improvement={delta:.3f} ft)")
    print(f"Gate 2 ve  rdict: {'PASS' if delta >= 0.5 else 'FAIL'} "
          f"(requires RMSE_model < RMSE_no-op - 0.5 ft)")

In [ ]:
def shuffled_toe_prediction(well_id: str, fold_model: LinearTreeExtrapolator, rng: np.random.Generator):
    """Rebuilds a well's Toe features with the GR/deltaZ sequence randomly permuted before
    Viterbi tracking — each row keeps its own X, Y, GR, and true TVT together (per Gate 1's
    'shuffle rows' procedure), but MD-sequential continuity is destroyed, so a Viterbi
    baseline computed on this permuted order is physically meaningless. Returns
    (pred_tvt, true_tvt) for the Toe rows, aligned to the permuted row order."""
    horiz = preprocess_horizontal(load_horizontal(well_id, "train"))
    typewell = load_typewell(well_id, "train")
    if len(horiz) < 5 or len(typewell) < 5:
        return None

    inc_deg, az_deg = derive_inclination_azimuth(horiz)
    horiz["tortuosity_cum"] = calculate_q3d_tortuosity(horiz["MD"].to_numpy(), inc_deg, az_deg)
    az_sin, az_cos = well_azimuth_sin_cos(horiz)
    horiz["azimuth_sin"], horiz["azimuth_cos"] = az_sin, az_cos
    horiz["dcor_gr_match"] = dcor_sliding_feature(horiz["GR"].to_numpy(), typewell["GR"].to_numpy())

    heel_frac = _deterministic_heel_fraction(well_id)
    toe_start = max(2, int(len(horiz) * heel_frac))
    if toe_start >= len(horiz) - 1:
        return None
    heel_end_tvt = float(horiz["TVT"].to_numpy()[toe_start - 1])

    z = horiz["Z"].to_numpy()
    delta_z_full = np.diff(z, prepend=z[0])
    toe_gr = horiz["GR"].to_numpy()[toe_start:]
    toe_dz = delta_z_full[toe_start:]

    perm = rng.permutation(len(toe_gr))
    shuffled_gr = toe_gr[perm]
    shuffled_dz = toe_dz[perm]

    tracker = AdvancedViterbiTracker(
        typewell_tvt=typewell["TVT"].to_numpy(), typewell_gr=typewell["GR"].to_numpy(),
        anchor_tvt=heel_end_tvt,
    )
    shuffled_viterbi = tracker.track(shuffled_gr, shuffled_dz, heel_end_tvt)

    toe_frame = horiz.iloc[toe_start:].reset_index(drop=True).iloc[perm].reset_index(drop=True)
    toe_frame["viterbi_tvt"] = shuffled_viterbi
    toe_frame["MD_from_ps"] = toe_frame["MD"] - horiz["MD"].to_numpy()[toe_start]

    X = clean_feature_matrix(toe_frame, FEATURE_COLS)
    pred_tvt = toe_frame["viterbi_tvt"].to_numpy() + fold_model.predict(X)
    return pred_tvt, toe_frame["TVT"].to_numpy()


if RUN_DIAGNOSTICS:
    print("=" * 70)
    print("GATE 1 — Sequence Shuffle Test")
    print("=" * 70)

    rng = np.random.default_rng(RANDOM_STATE)
    SHUFFLE_TEST_N_WELLS = 60
    sample_wells = rng.choice(train_well_ids, size=min(SHUFFLE_TEST_N_WELLS, len(train_well_ids)), replace=False)

    shuffled_preds, shuffled_trues = [], []
    for well_id in sample_wells:
        cluster_id = well_to_cluster.get(well_id)
        fold_model = fold_models.get(cluster_id)
        if fold_model is None:
            continue
        result = shuffled_toe_prediction(well_id, fold_model, rng)
        if result is None:
            continue
        pred, true = result
        shuffled_preds.append(pred)
        shuffled_trues.append(true)

    rmse_shuffled = rmse(np.concatenate(shuffled_preds), np.concatenate(shuffled_trues))

    sample_mask = train_toe_df["well_id"].isin(sample_wells) & train_toe_df["well_id"].map(well_to_cluster).map(
        lambda c: fold_models.get(c) is not None
    )
    rmse_seq_sample = rmse(oof_pred[sample_mask.to_numpy()], true_tvt[sample_mask.to_numpy()])

    leakage_index = rmse_shuffled / rmse_seq_sample if rmse_seq_sample > 0 else np.nan
    print(f"RMSE_sequential (same well sample): {rmse_seq_sample:.3f} ft")
    print(f"RMSE_shuffled:                      {rmse_shuffled:.3f} ft")
    print(f"Leakage Index = RMSE_shuffled / RMSE_sequential = {leakage_index:.2f}")
    if leakage_index >= 2.5:
        verdict = "APPROVED (model relies on genuine sequential GR alignment)"
    elif leakage_index <= 1.2:
        verdict = "FAIL (model ignores sequence order — likely coordinate leakage)"
    else:
        verdict = "INCONCLUSIVE (between the documented fail/approve thresholds)"
    print(f"Gate 1 verdict: {verdict}")

## Inference: dynamically-globbed test wells → `submission.csv`

Uses the residual model trained on **all** training Toe rows (`residual_model`, not a CV
fold model) since this is the real deployment path. For every test well found by
`list_well_ids("test")` — which will be the hidden set of "hundreds of unseen wells" at
scoring time, not the 3 public placeholders — this: builds features with `build_well_frame`,
predicts `viterbi_tvt + residual_model.predict(...)` for every row where `TVT_input` is NaN,
and assembles `id = {WELLNAME}_{row_index}` / `tvt` rows into `submission.csv`.

In [ ]:
def predict_test_well(well_id: str) -> pd.DataFrame:
    """Returns a DataFrame with columns [orig_index, tvt] for every row where TVT_input is NaN
    in this test well. Falls back to a no-op (Heel-end hold) prediction if the well is too
    degenerate for the full feature/Viterbi pipeline (e.g. too few stations) — every required
    id must get a prediction, even in that edge case."""
    raw = load_horizontal(well_id, "test")
    nan_mask_raw = raw["TVT_input"].isna().to_numpy()
    if not nan_mask_raw.any():
        return pd.DataFrame(columns=["orig_index", "tvt"])

    try:
        frame = build_well_frame(well_id, "test")
    except Exception as exc:
        print(f"  [warn] {well_id}: pipeline failed ({exc}); falling back to no-op hold.")
        frame = None

    if frame is not None:
        toe = frame[frame["is_toe"]].copy()
        X = clean_feature_matrix(toe, FEATURE_COLS)
        pred_residual = residual_model.predict(X)
        toe["tvt_pred"] = toe["viterbi_tvt"].to_numpy() + pred_residual
        return toe[["orig_index", "tvt_pred"]].rename(columns={"tvt_pred": "tvt"})

    # Fallback: hold the last known TVT_input constant across the masked rows.
    known = raw.loc[~nan_mask_raw].sort_values("MD")
    heel_end_tvt = float(known["TVT_input"].iloc[-1]) if len(known) else 0.0
    fallback_idx = raw.loc[nan_mask_raw, "orig_index"].to_numpy()
    return pd.DataFrame({"orig_index": fallback_idx, "tvt": heel_end_tvt})


submission_rows = []
for well_id in test_well_ids:
    well_pred = predict_test_well(well_id)
    well_pred["id"] = well_pred["orig_index"].apply(lambda idx: f"{well_id}_{idx}")
    submission_rows.append(well_pred[["id", "tvt"]])

submission_df = pd.concat(submission_rows, ignore_index=True) if submission_rows else pd.DataFrame(columns=["id", "tvt"])
submission_df.to_csv("submission.csv", index=False)
print(f"submission.csv written with {len(submission_df)} predicted rows across {len(test_well_ids)} test wells.")
submission_df.head()